# Lượt 4 — PaddleOCR + VietOCR trên toàn batch 1 → Volume `keyframes-btc`

`config/ingest_shards_batch1_btc_ocr2.yaml` · chạy **sau** `02-keyframes-btc-import.ipynb`

**Chỉ Trọng và Quang.** Cùng bundle, cùng volume, **cùng bảng phân công** như ba lượt trước —
đổi `MEMBER` ở ô đầu là ra đúng phần việc quen thuộc.

| tầng | artifact | trên Volume | phát hiện | nhận dạng |
|---|---|---|---|---|
| `n5_ocr_paddle` | `paddle_ocr` | `parts/paddle_ocr/` | PaddleOCR | PaddleOCR `lang='vi'` |
| `n5_ocr_vietocr` | `viet_ocr` | `parts/viet_ocr/` | **PaddleOCR** (dùng lại) | VietOCR |

### Vì sao giữ CẢ HAI, không chọn một

Đo trên 40 keyframe chung của `L21_V001`, **cùng bộ phát hiện**, khác đúng bộ nhận dạng:

| | chữ có dấu | tiêu đề tiếng Việt | logo đài |
|---|---|---|---|
| **VietOCR** | **20,6%** | `TÌNH TRẠNG SỤT LÚN Ở ĐBSCL ĐANG DIỄN RA RẤT NHANH` | `HTVập` `HTM2` `HIVE` |
| **PaddleOCR** | 10,7% | `TINH TRANG SUT LÚN dBSCL dANG DIN RA RÃT NHANH` | `HTV9` sạch |

Hai bộ sai ở **hai chỗ khác nhau**. VietOCR đọc tiêu đề tiếng Việt gần như hoàn hảo nhưng biến
logo thành từ tiếng Việt; PaddleOCR đọc logo/đồng hồ sạch nhưng rụng dấu có hệ thống. Fusion ăn
được cả hai — đó là phép cộng, không phải phép chọn (§8.6 handoff).

> ⚠️ Số trên đo trên **40 keyframe của một video tin tức**. Nó đủ để biện minh cho việc SINH cả
> hai artifact; nó **không** đủ để đặt trọng số. Trọng số đặt bằng leo dốc trên tập dev và công
> bố trên heldout (§9.4) — sau khi lượt này xong.

### Vòng lặp theo VIDEO, không theo TẦNG — khác notebook 03

`n5_ocr_vietocr` dùng **chính** bộ phát hiện của `n5_ocr_paddle`, và `paddle_scan` giữ kết quả
lại trong phiên. Chạy hai tầng liền nhau trên từng video thì VietOCR **không phải trả lại giá của
cả pipeline PaddleOCR** — đo được trên CPU: đó là một nửa tổng thời gian.

Cache chỉ giữ **một video một lúc** (chặn bộ nhớ: 400 video × ~300 keyframe sẽ ăn hết RAM sau
nhiều giờ), nên đảo thứ tự thành theo-tầng là mất sạch lợi ích đó.

### KHÔNG gọi `unload_models()` giữa các video

PaddleOCR 3.x khởi tạo PaddleX ở phạm vi **tiến trình**. Thả model rồi dựng lại trong cùng phiên
là chết bằng `RuntimeError: PDX has already been initialized` — không phải OOM, không phải lỗi
tạm. Notebook này chỉ thả model ở ô cuối cùng.

### Cần có trước khi chạy

| | |
|---|---|
| Kaggle Secrets | `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET`, `GITHUB_TOKEN` (bắt buộc nếu không attach dataset `mfe-repo`) |
| Settings | Internet **On** · Accelerator **GPU T4** |
| Dataset | gói `Keyframes_*.zip` của bạn (không có thì notebook tự tải, 1,4–6 GB có resume) |

**GPU không phải tuỳ chọn ở lượt này.** Đo 19-08 trên CPU với oneDNN đã tắt: một video 307
keyframe chạy **hơn 25 phút chưa xong**. Ở quy mô 404–469 video thì đường CPU không dùng được.

## Step 0a — repo + môi trường nền

In [ ]:
MEMBER      = "trong"          # Đổi duy nhất dòng này: "trong" hoặc "quang"
NOTEBOOK_BUILD = "nb06-paddle322-cu126-r8"
print(f"NOTEBOOK BUILD: {NOTEBOOK_BUILD}")
SHARDS      = "config/ingest_shards_batch1_btc_ocr2.yaml"
TRANSPORT   = "modal"
VOLUME      = "keyframes-btc"  # CÙNG volume với notebook 02/03/04
VOL_VERSION = 2

# ── CHẠY THỬ / CHẠY LẠI ───────────────────────────────────────────────────
# ONLY_VIDEOS: [] = cả phần việc của bạn. Điền video_id để chỉ chạy bấy nhiêu.
#              Lần chạy ĐẦU nên để 1 video, xem s/ảnh rồi mới thả cả gói.
# REDO_STAGES: chạy lại DÙ marker đã báo xong. GHI ĐÈ part + marker cũ.
# LIMIT:       số ảnh mỗi video; 0 = tất cả. Chỉ dùng để thử, KHÔNG dùng khi
#              chạy thật — một video bị cắt vẫn cắm marker "xong".
ONLY_VIDEOS = []
REDO_STAGES = []
LIMIT       = 0
# Tương thích với lượt n0 cũ: một số marker keyframe nằm ở bundle cũ,
# trong khi thư mục keyframe trên Volume đã có đủ. Không cắt video nào.
ALLOW_KEYFRAME_DIR_FALLBACK = True

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/triet/structure"

import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/input/mfe-repo")
# Chỉ dùng dataset đính kèm nếu nó thực sự chứa mã `src/mfe`.
if not (REPO / "src" / "mfe").is_dir():
    REPO = Path("/kaggle/working/mfe")
    if not (REPO / "src" / "mfe").is_dir():
        # Repo private: lấy token từ Kaggle Secret và truyền qua môi trường
        # Git, không nhét token vào URL/traceback/log.
        # Dọn clone dở từ lần chạy trước, chỉ trong /kaggle/working/mfe.
        if REPO.exists():
            if REPO.is_dir() and not REPO.is_symlink():
                shutil.rmtree(REPO)
            else:
                REPO.unlink()
        from kaggle_secrets import UserSecretsClient
        try:
            github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            raise RuntimeError(
                "Thiếu Kaggle Secret GITHUB_TOKEN. Repo là private; "
                "phải thêm secret này rồi bật quyền cho notebook."
            ) from None
        if not github_token:
            raise RuntimeError("Kaggle Secret GITHUB_TOKEN đang rỗng.")
        clone_env = os.environ.copy()
        clone_env["GIT_TERMINAL_PROMPT"] = "0"
        clone_env["GIT_CONFIG_COUNT"] = "1"
        clone_env["GIT_CONFIG_KEY_0"] = "http.extraHeader"
        clone_env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: bearer {github_token}"
        url = REPO_URL
        clone = subprocess.run(
            ["git", "clone", "--depth", "1", "--single-branch",
             "-b", REPO_REF, url, str(REPO)],
            text=True, capture_output=True, env=clone_env,
        )
        if clone.returncode != 0:
            detail = (clone.stderr or clone.stdout or "không có stderr").strip()
            detail = detail.replace(github_token, "***")
            raise RuntimeError(
                f"Clone repo private thất bại (branch={REPO_REF}). Kiểm tra Internet On và GITHUB_TOKEN; chi tiết git: {detail[-2000:]}"
            )
        del github_token, clone_env
sys.path.insert(0, str(REPO / "src"))

# Bản .lock thắng bản dải phiên bản — năm người phải cùng phiên bản (§2.5).
req = REPO / "notebooks" / "requirements-kaggle.lock.txt"
if not req.is_file():
    req = REPO / "notebooks" / "requirements-kaggle-btc-images.txt"
print(f"môi trường nền: {req.name}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)

## Step 0b — PaddleOCR + VietOCR

> ⚠️ **Ô này đã hai lần làm hỏng lần chạy, theo hai cách khác nhau.** Cả hai đều
> im lặng — không lỗi nào được ném, và cả hai chỉ lộ ra khi đọc con số thời gian.
>
> **Lần 1 — `-i` thay thế PyPI.** Cài `paddlepaddle-gpu` bằng `-i <index Paddle>`
> khiến pip giải *toàn bộ* phụ thuộc từ index đó, hạ `nvidia-nccl-cu12` xuống dưới
> bản torch cần rồi vẫn thất bại. Kết quả: `libtorch_cuda.so: undefined symbol:
> ncclCommShrink`, lộ ra ở tầng OCR hai ô sau. `--no-deps` chặn điều đó.
>
> **Lần 2 — bản CPU đè bản GPU.** File requirements khai `paddlepaddle` (CPU) làm
> "đường lui", nhưng notebook cài GPU **trước** rồi mới đọc file đó — mà hai gói
> cùng cài ra một thư mục `paddle/`. Bản CPU luôn thắng. Đo được: **10,5 giây/ảnh**
> thay vì dưới 0,5 trên T4, tức gần bằng Qwen2-VL (10,8) — chính cái mà lượt này
> sinh ra để thay thế.

**Thứ tự hiện tại**: requirements trước → gỡ **mọi** bản paddle → cài đúng **một**
bản sau cùng → đếm lại, còn hai bản thì dừng hẳn.

**Ràng buộc `-c` gồm cả `nvidia-*`.** Không có nó, pip được phép đổi torch để thoả
mãn `vietocr`, và tiền xử lý ảnh của SigLIP đổi theo — kiểu lệch **không ném lỗi**,
nó chỉ làm bundle thành hỗn hợp hai phép đo (§2.5).

In [ ]:
from importlib.metadata import PackageNotFoundError, version

# Không thể gỡ an toàn CUDA shared libraries khỏi một Python process đã nạp
# Torch/Paddle. Dừng TRƯỚC khi thay wheel để lỗi không xuất hiện muộn ở Step 0c.
already_loaded = [name for name in ("torch", "paddle") if name in sys.modules]
if already_loaded:
    raise RuntimeError(
        "KERNEL BAN: da nap " + ", ".join(already_loaded) +
        ". Chon Factory Reset Session, upload R8, roi Run All tu dau; "
        "khong chay lai rieng cell nay."
    )

# ⚠️ THỨ TỰ CÀI Ở Ô NÀY LÀ MỘT SỬA LỖI, KHÔNG PHẢI SỞ THÍCH.
#
# Bản trước cài `paddlepaddle-gpu` TRƯỚC rồi mới `pip install -r requirements`,
# mà file requirements khai `paddlepaddle` (bản CPU) làm "đường lui". Hai gói đó
# **cùng cài ra một thư mục `paddle/`**, nên lần cài sau GHI ĐÈ lần trước: bản
# GPU luôn bị bản CPU đè, mọi lần chạy.
#
# Đo được hậu quả 19-08-2026: PaddleOCR chạy **10,5 giây/ảnh** — gần bằng
# Qwen2-VL (10,8), trong khi trên T4 nó phải dưới 0,5. Không có lỗi nào được ném,
# `is_compiled_with_cuda()` vẫn trả về đúng cho bản đang cài, và ô kiểm môi
# trường vẫn in "OK".
#
# Giờ: requirements TRƯỚC, gỡ sạch mọi bản paddle, rồi cài đúng MỘT bản sau cùng.
ALLOW_CPU        = False     # True = chấp nhận chạy CPU (rất chậm — xem ô kiểm)
# Dùng wheel Paddle GPU cu126 tự chứa runtime, cài --no-deps để không đổi
# bộ nvidia-* CUDA 12.8 của torch. Wheel cu128 3.2.0 đã lỗi ABI nội bộ
# (_xpu_get_current_stream) trên image Kaggle hiện tại nên không dùng.
PADDLE_GPU_WHEEL = (
    "https://paddle-whl.bj.bcebos.com/stable/cu126/paddlepaddle-gpu/"
    "paddlepaddle_gpu-3.2.2-cp312-cp312-linux_x86_64.whl"
)
PADDLE_GPU_INDEX = None

# Ràng buộc: gói mà đổi phiên bản là đổi SỐ ĐO, cộng các gói CUDA mà ABI của
# torch phụ thuộc vào. `nvidia-*` có mặt vì chính chúng là thứ đã bị hạ lần trước.
pins = []
for package in ("torch", "torchvision", "transformers", "numpy", "pillow",
                "nvidia-nccl-cu12", "nvidia-cublas-cu12", "nvidia-cudnn-cu12",
                "nvidia-cuda-runtime-cu12"):
    try:
        pins.append(f"{package}=={version(package)}")
    except PackageNotFoundError:
        continue
constraints = Path("/kaggle/working/paddle-constraints.txt")
constraints.write_text(chr(10).join(pins) + chr(10), encoding="utf-8")
print("ràng buộc:")
for pin in pins:
    print(f"  {pin}")

# 1. paddleocr + vietocr. Có thể kéo theo `paddlepaddle` CPU như phụ thuộc —
#    bước 2 dọn sạch nên không sao.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-c", str(constraints),
     "-r", str(REPO / "notebooks" / "requirements-kaggle-btc-paddle-vietocr.txt")],
    check=True,
)

# 2. Gỡ MỌI bản paddle. Không có bước này thì bước 3 chỉ chồng lên một cây thư
#    mục `paddle/` đã có, và cái nào thắng là chuyện của thứ tự file.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                "paddlepaddle", "paddlepaddle-gpu"])

# Xoá phần còn sót của các wheel Paddle CPU/GPU đã chồng lên nhau.
# Nếu không dọn, import paddle có thể lấy module dở từ lần cài trước.
import site, shutil
for site_root in site.getsitepackages():
    site_path = Path(site_root)
    shutil.rmtree(site_path / "paddle", ignore_errors=True)
    for pattern in ("paddlepaddle*.dist-info", "paddlepaddle_gpu*.dist-info"):
        for dist_path in site_path.glob(pattern):
            shutil.rmtree(dist_path, ignore_errors=True)
for module_name in list(sys.modules):
    if module_name == "paddle" or module_name.startswith("paddle."):
        del sys.modules[module_name]

# 3. Đúng MỘT bản paddle, cài sau cùng. `--no-deps` chặn nó chạm vào `nvidia-*`
#    của torch — đúng cái đã phá torch bằng `undefined symbol: ncclCommShrink`.
paddle_install = [sys.executable, "-m", "pip", "install", "-q",
                  "--no-deps", PADDLE_GPU_WHEEL]
if PADDLE_GPU_INDEX:
    paddle_install += ["-i", PADDLE_GPU_INDEX]
gpu_ok = subprocess.run(paddle_install).returncode == 0
if not gpu_ok:
    raise RuntimeError("Khong cai duoc Paddle cu126 GPU; dung lai, khong fallback CPU.")

# Paddle vẫn import `astor`, nhưng dependency này không được kéo khi wheel
# theo khi GPU wheel được cài bằng --no-deps. Cài rõ trước ô kiểm GPU.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--no-deps", "astor==0.8.1",
                "opt_einsum==3.3.0"], check=True)

# Kiểm bằng process Python SẠCH, đúng thứ tự Paddle -> Torch. Phép thử cuối
# quay lại Paddle để bắt xung đột thư viện CUDA giữa hai framework.
probe_code = (
    "import paddle; "
    "paddle.set_device('gpu'); "
    "a=paddle.ones([64,64]); "
    "assert float(a.sum().item())==4096.0; "
    "import torch; "
    "assert torch.cuda.is_available(); "
    "b=torch.ones((64,64),device='cuda'); "
    "assert float(b.sum().item())==4096.0; "
    "c=paddle.ones([32,32]); "
    "assert float(c.sum().item())==1024.0; "
    "print('paddle='+paddle.__version__, paddle.device.get_device(), "
    "      'torch='+torch.__version__, torch.cuda.get_device_name(0))"
)
probe = subprocess.run([sys.executable, "-c", probe_code],
                       text=True, capture_output=True)
print("Paddle->Torch fresh-process probe:", probe.stdout.strip())
if probe.returncode != 0:
    print(probe.stderr)
    raise RuntimeError("Paddle/Torch GPU probe that bai trong process sach.")

# Nạp Paddle NGAY trong kernel chính. Từ đây Torch chỉ được phép nạp sau Paddle.
# Đây là phần R6 còn thiếu: process phụ sạch không bảo vệ thứ tự của kernel chính.
if "torch" in sys.modules:
    raise RuntimeError("Torch bi nap vao kernel trong luc cai dat; Factory Reset Session.")
import paddle
paddle.set_device("gpu")
main_value = float(paddle.ones([64, 64]).sum().item())
assert main_value == 64 * 64, main_value
print(f"Paddle main-kernel ready: {paddle.__version__} {paddle.device.get_device()}")
PADDLE_MAIN_READY = True
DUAL_GPU_PROBE_OK = True

# Phải còn ĐÚNG MỘT. Hai bản cùng có nghĩa là bước 2 trượt, và bản nào thực sự
# chạy lại là chuyện của thứ tự file — im lặng, và chậm gấp 20 lần.
builds = []
for name in ("paddlepaddle", "paddlepaddle-gpu"):
    try:
        builds.append(f"{name}=={version(name)}")
    except PackageNotFoundError:
        continue
print()
print("paddle đã cài: " + (", ".join(builds) or "(khong co)"))
if len(builds) != 1:
    raise SystemExit(f"phai co dung MOT ban paddle, dang co {len(builds)}: {builds}")

## Step 0c — kiểm môi trường (chạy trước, ~20 giây)

Ô này **chạy thật một phép tính trên GPU**, không hỏi `is_compiled_with_cuda()`.
Hàm đó chỉ nói bản đang cài có được *build* với CUDA hay không — nó trả về đúng cho
bản CPU (False) và cho bản GPU không dùng được (True). Chính vì hỏi nó mà lần chạy
10,5 giây/ảnh vẫn in ra "môi trường OK".

Mặc định **dừng hẳn nếu không có GPU**. Đặt `ALLOW_CPU = True` ở ô trên nếu thật sự
muốn chạy CPU — nhớ giảm số ảnh trước.

In [ ]:
# KIỂM MÔI TRƯỜNG — phải chạy trước Step 2, không phải để bảo hiểm.
#
# Ô này CHẠY THẬT một phép tính trên GPU chứ không hỏi `is_compiled_with_cuda()`.
# Hàm đó chỉ nói bản đang cài có được BUILD với CUDA hay không — nó trả về đúng
# cho bản CPU (False) và cho bản GPU không dùng được (True), nên nó đã để lọt
# đúng lần chạy 10,5 giây/ảnh mà không ai biết cho tới khi đọc log tiến độ.
os.environ.setdefault("FLAGS_use_mkldnn", "0")

EXPECTED_BUILD = "nb06-paddle322-cu126-r8"
if globals().get("NOTEBOOK_BUILD") != EXPECTED_BUILD:
    raise RuntimeError("Dang chay notebook cu/sai build. Upload notebook 06 R8 va Run All tu dau.")
if not globals().get("DUAL_GPU_PROBE_OK", False):
    raise RuntimeError("Step 0b R8 chua probe Paddle->Torch GPU thanh cong; khong duoc chay rieng Step 0c.")
if not globals().get("PADDLE_MAIN_READY", False):
    raise RuntimeError("Step 0b R8 chua nap Paddle vao kernel chinh. Factory Reset va Run All.")

problems = []
DEVICE = "cpu"
try:
    import paddle
    print(f"paddle {paddle.__version__} · compiled_with_cuda="
          f"{paddle.device.is_compiled_with_cuda()}")
    try:
        paddle.set_device("gpu")
        # Ép thực thi: `.item()` buộc kernel chạy thật trên thiết bị.
        value = float((paddle.zeros([64, 64]) + 1).sum().item())
        assert value == 64 * 64, value
        DEVICE = "gpu"
        print(f"paddle chay that tren {paddle.device.get_device()}")
    except Exception as err:
        paddle.set_device("cpu")
        problems.append(f"paddle KHONG chay duoc tren GPU: {type(err).__name__}: {err}")
except Exception as err:
    problems.append(f"paddle KHONG import duoc: {err}")

TORCH_DEVICE = "cpu"
try:
    # QUAN TRỌNG: chỉ nạp Torch sau khi Paddle đã nạp và chạy kernel GPU.
    import torch
    print(f"torch {torch.__version__} · cuda={torch.cuda.is_available()}")
    if not torch.cuda.is_available():
        problems.append("torch KHONG thay CUDA — VietOCR se chay CPU")
    else:
        torch_value = float(torch.ones((64, 64), device="cuda").sum().item())
        assert torch_value == 64 * 64, torch_value
        TORCH_DEVICE = "gpu"
        print(f"torch chay that tren {torch.cuda.get_device_name(0)}")
except Exception as err:
    problems.append(f"torch KHONG chay duoc tren GPU: {type(err).__name__}: {err}")

if DEVICE == "gpu" and TORCH_DEVICE == "gpu":
    try:
        # Kiểm Paddle thêm lần nữa sau khi Torch đã nạp CUDA libraries.
        paddle_value = float(paddle.ones([32, 32]).sum().item())
        assert paddle_value == 32 * 32, paddle_value
        print("paddle van chay sau khi nap torch")
    except Exception as err:
        DEVICE = "cpu"
        problems.append(f"paddle HONG sau khi nap torch: {type(err).__name__}: {err}")

for package in ("paddleocr", "vietocr"):
    try:
        print(f"{package} {version(package)}")
    except PackageNotFoundError:
        problems.append(f"{package} chua cai")

if DEVICE == "cpu":
    problems.append(
        "paddle chay CPU. Do 19-08-2026: 10,5 giay/anh — gan bang Qwen2-VL "
        "(10,8), trong khi tren T4 no phai duoi 0,5. Mot video 307 keyframe "
        "mat ~54 phut chi cho tang paddle."
    )

if problems:
    print()
    for line in problems:
        print(f"  HONG: {line}")
    print()
    print("Khong tu pip install/downgrade cac goi nvidia-* trong kernel nay.")
    print("Neu Paddle/libpaddle lech API, Factory reset roi Run All tu dau.")
    if not ALLOW_CPU:
        raise SystemExit(
            "moi truong GPU chua san sang; dung pipeline va gui toan bo log Step 0b/0c."
        )

print()
print(f"moi truong OK · paddle={DEVICE} · torch={TORCH_DEVICE}")

## Step 0d — Modal, phân công, việc đã xong

`StageStatus` đọc marker **một lần** cho cả phiên. Marker là điểm chốt, không phải file dữ liệu:
một part đẩy dở vẫn tồn tại, nên "có file" không thể có nghĩa là "đã xong" (§5.2).

In [ ]:
import logging, yaml
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from kaggle_secrets import UserSecretsClient
from mfe.offline.bundle import BundleLayout, BundleWriter, StageStatus, load_shard_plan
from mfe.offline.stages import STAGES, StageContext, read_keyframes, unload_models
from mfe.offline.transport import build_transport

secrets = UserSecretsClient()
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")

RAW       = yaml.safe_load((REPO / SHARDS).read_text(encoding="utf-8"))
plan      = load_shard_plan(REPO / SHARDS)
layout    = BundleLayout(plan.bundle)
transport = build_transport(TRANSPORT, volume=VOLUME,
                            local_root="/kaggle/working/bundle_local",
                            version=VOL_VERSION)
WORK = Path("/kaggle/working/scratch"); WORK.mkdir(parents=True, exist_ok=True)
ZIPS = Path("/kaggle/working/zips");    ZIPS.mkdir(parents=True, exist_ok=True)
KAGGLE_INPUT = Path("/kaggle/input")
writer = BundleWriter(transport, layout, MEMBER, WORK)

assert plan.keyframe_set == "btc"
assert plan.stages == ["n5_ocr_paddle", "n5_ocr_vietocr"], plan.stages
STAGE_LIST = plan.stages
EXTRAS     = RAW["extras"]

# Tham số từ yaml, KHÔNG khai lại ở đây. `device` và `limit` là hai thứ duy nhất
# notebook được quyết, vì chúng phụ thuộc phiên chứ không phải phép đo.
PARAMS = {}
for stage in STAGE_LIST:
    PARAMS[stage] = {**RAW["params"].get(stage, {}), "device": DEVICE, "limit": LIMIT}

MY_ZIPS   = plan.zips_for(MEMBER)
MY_VIDEOS = plan.videos_for(MEMBER)
if ONLY_VIDEOS:
    invalid = sorted(set(ONLY_VIDEOS) - set(MY_VIDEOS))
    if invalid:
        raise ValueError(
            f"ONLY_VIDEOS co video khong thuoc phan cua {MEMBER}: {invalid}. "
            "Quang thuoc L22/L26_a/L26_c/L26_d/L26_e/L27/L28; Trọng thuoc L21/L23/L24/L25/L26_b/L29/L30."
        )
status    = StageStatus.load(transport, layout, MEMBER)

# Điều kiện tiên quyết: notebook 02 phải xong TRƯỚC.
imported = StageStatus.load(transport, layout, MEMBER)
keyframe_dirs = set()
if ALLOW_KEYFRAME_DIR_FALLBACK:
    try:
        keyframe_dirs = {Path(str(remote)).name for remote in
                         transport.listdir(f"{layout.root}/keyframes")}
    except Exception as err:
        print(f"không đọc được thư mục keyframe để dự phòng: {err}")

def has_n0_import(video_id):
    return (imported.is_done("n0_import_btc", video_id) or
            (ALLOW_KEYFRAME_DIR_FALLBACK and video_id in keyframe_dirs))

missing_import = [v for v in MY_VIDEOS if not has_n0_import(v)]
todo = [(s, v) for v in MY_VIDEOS for s in STAGE_LIST if not status.is_done(s, v)]

print(f"bundle  : {plan.bundle}  ->  volume {VOLUME} (v{VOL_VERSION})")
print(f"người   : {MEMBER}")
print(f"zip     : {', '.join(n for n, _ in MY_ZIPS)}")
print(f"video   : {len(MY_VIDEOS)} (tổng {len(plan.videos)})")
print(f"tầng    : {', '.join(STAGE_LIST)}")
print(f"còn lại : {len(todo)} việc / {len(MY_VIDEOS) * len(STAGE_LIST)}")
print(f"thiết bị: {DEVICE}")
for stage, body in PARAMS.items():
    print(f"  {stage:<16} {body}")
if ONLY_VIDEOS:
    print(f"CHẾ ĐỘ THỬ : chỉ {len(ONLY_VIDEOS)} video -> {', '.join(ONLY_VIDEOS)}")
if REDO_STAGES:
    print(f"CHẠY LẠI   : {', '.join(REDO_STAGES)} (ghi đè part + marker cũ)")
if LIMIT:
    print(f"CẢNH BÁO   : LIMIT={LIMIT} — marker vẫn báo 'xong' cho video bị cắt.")
if missing_import:
    print()
    print(f"CHU Y: {len(missing_import)} video CHUA nhap bang notebook 02 — se BO QUA.")
    print("   " + ", ".join(missing_import[:6]) + (" …" if len(missing_import) > 6 else ""))

## Step 1 — ảnh keyframe

Ưu tiên Kaggle Dataset đã attach (đã giải nén hoặc còn ZIP nguyên bản); thiếu cả hai mới tải.
Chỉ bung **một video** mỗi lượt — bung cả gói `Keyframes_L26_*` là nhiều GB nằm chờ vô ích.

`--read-timeout=60` là thứ quan trọng nhất trong lệnh `wget`: không có nó, một kết nối bị bóp
xuống vài trăm B/s vẫn là kết nối "sống" và wget đợi vô hạn. Mỗi gói được **kiểm kích thước**
sau khi tải, vì `wget -c` gặp máy chủ lờ `Range` sẽ nối cả body vào sau phần đã có.

In [ ]:
import shutil, zipfile

def free_gb():
    return shutil.disk_usage("/kaggle/working").free / 2**30


def validate_zip(path):
    try:
        with zipfile.ZipFile(path) as archive:
            if not archive.infolist():
                raise RuntimeError(f"{path}: ZIP rong")
    except zipfile.BadZipFile as err:
        raise RuntimeError(f"{path}: khong phai ZIP hoan chinh") from err


def find_kaggle_zip(name, expect_bytes=None):
    if not KAGGLE_INPUT.is_dir():
        return None
    found = sorted(p for p in KAGGLE_INPUT.rglob(name) if p.is_file())
    if not found:
        return None
    if len(found) != 1:
        raise RuntimeError(f"{name}: thay nhieu ban: {', '.join(str(p) for p in found)}")
    actual = found[0].stat().st_size
    if expect_bytes is not None and actual != expect_bytes:
        raise RuntimeError(f"{found[0]}: {actual:,} byte, cho doi {expect_bytes:,}")
    validate_zip(found[0])
    print(f"    cache Kaggle: {found[0]} ({actual / 2**30:.2f} GB)")
    return found[0]


def find_kaggle_extracted(archive_name, child):
    stem = Path(archive_name).stem
    if not KAGGLE_INPUT.is_dir():
        return None
    for root, directories, _files in os.walk(KAGGLE_INPUT):
        if stem in directories:
            extracted = Path(root) / stem / child
            if not extracted.is_dir():
                raise RuntimeError(f"{Path(root) / stem}: thieu thu muc {child}")
            print(f"    cache Kaggle da giai nen: {extracted}")
            return extracted
        directories[:] = [d for d in directories if d != stem]
    return None


def download(url, name, expect_bytes=None):
    cached = find_kaggle_zip(name, expect_bytes)
    if cached is not None:
        return cached
    target = ZIPS / name
    if target.is_file() and expect_bytes is not None and target.stat().st_size == expect_bytes:
        validate_zip(target)
        print(f"    {name}: da co ({target.stat().st_size / 2**30:.2f} GB)")
        return target
    print(f"    tai {name} … (dia trong {free_gb():.1f} GB)")
    subprocess.run(
        ["wget", "-q", "--show-progress", "-c",
         "--timeout=30", "--read-timeout=60", "--tries=0", "--waitretry=10",
         "--retry-connrefused", "-O", str(target), url],
        check=True,
    )
    if expect_bytes is not None and target.stat().st_size != expect_bytes:
        size = target.stat().st_size
        target.unlink(missing_ok=True)
        raise RuntimeError(f"{name}: tai ve {size:,} byte, cho doi {expect_bytes:,}. Da xoa.")
    validate_zip(target)
    return target


def extract_video(zip_path, video_id, into):
    out = into / video_id
    out.mkdir(parents=True, exist_ok=True)
    count = 0
    with zipfile.ZipFile(zip_path) as archive:
        for name in archive.namelist():
            parts = Path(name).parts
            if len(parts) < 2 or parts[-2] != video_id or name.endswith("/"):
                continue
            with archive.open(name) as src, (out / parts[-1]).open("wb") as dst:
                shutil.copyfileobj(src, dst, length=1 << 20)
            count += 1
    return out, count


def extract_one_file(zip_path, inner_suffix, dest):
    with zipfile.ZipFile(zip_path) as archive:
        match = [n for n in archive.namelist() if n.endswith(inner_suffix)]
        if not match:
            raise FileNotFoundError(f"{inner_suffix} khong co trong {zip_path.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(match[0]) as src, dest.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    return dest


# `map-keyframes` lấy từ gói ~1 MB của BTC, KHÔNG kéo từ Volume: cùng một file,
# mà đường này không tốn lượt đọc Volume nào.
print("gói dùng chung:")
MAPS_DIR = find_kaggle_extracted("map-keyframes-aic25-b1.zip", "map-keyframes")
MAPS_ZIP = None if MAPS_DIR is not None else download(
    EXTRAS["map_keyframes"], "map-keyframes-aic25-b1.zip"
)
print(f"đĩa trống: {free_gb():.1f} GB")

## Step 2 — vòng lặp

Theo **video**, và trong mỗi video chạy **cả hai tầng liền nhau** — đó là điều kiện để
`paddle_scan` dùng lại kết quả phát hiện. Đảo thành theo-tầng là chạy PaddleOCR hai lượt.

`merge=True` khi ghi manifest: bundle này giờ được đổ bởi **bốn** notebook (02 nhập, 03 SigLIP+OCR,
04 video, 06 đây) và cả bốn cùng ghi `parts/manifest/{video}.json`. Không gộp thì mỗi lượt xoá
sạch artifact của lượt trước, còn marker vẫn báo "xong" nên **không lần chạy nào phục hồi lại**.

ETA tính từ số video **đã chạy thật trong phiên này**, không từ ước lượng nào viết sẵn.

In [ ]:
import time, traceback


def stages_for(video_id):
    return [s for s in STAGE_LIST
            if s in REDO_STAGES or not status.is_done(s, video_id)]


failures = []
started = time.perf_counter()
done_count = 0
images_seen = 0

for zip_name, zip_url in MY_ZIPS:
    mine = set(MY_VIDEOS)
    if ONLY_VIDEOS:
        mine &= set(ONLY_VIDEOS)
    videos = [v for v in plan.videos_in(zip_name)
              if v in mine and has_n0_import(v)]
    pending = [v for v in videos if stages_for(v)]
    if not pending:
        print()
        print(f"=== {zip_name}: xong hết, không tải ===")
        continue

    print()
    print(f"=== {zip_name}: còn {len(pending)}/{len(videos)} video ===")
    kf_root = find_kaggle_extracted(zip_name, "keyframes")
    kf_zip = None if kf_root is not None else download(
        zip_url, zip_name, plan.zip_bytes.get(zip_name)
    )

    for video_id in pending:
        stages = stages_for(video_id)
        try:
            if kf_root is not None:
                kf_dir = kf_root / video_id
                if not kf_dir.is_dir():
                    raise FileNotFoundError(f"{video_id}: thiếu trong {kf_root}")
                n_img = sum(1 for p in kf_dir.iterdir()
                            if p.is_file() and p.suffix.lower() == ".jpg")
            else:
                kf_dir, n_img = extract_video(kf_zip, video_id, WORK / "keyframes")
            if MAPS_DIR is not None:
                map_csv = MAPS_DIR / f"{video_id}.csv"
                if not map_csv.is_file():
                    raise FileNotFoundError(f"{map_csv}: không tồn tại")
            else:
                map_csv = extract_one_file(MAPS_ZIP, f"/{video_id}.csv",
                                           WORK / "map-keyframes" / f"{video_id}.csv")

            keyframes = read_keyframes(map_csv, kf_dir)
            print(f"  {video_id}  {n_img} ảnh  ({len(stages)} tầng)")
            ctx = StageContext(video_id, Path("/dev/null"), keyframes,
                               writer, layout, WORK)
            markers = []
            for stage in stages:
                ctx.params = dict(PARAMS.get(stage, {}))
                marker = STAGES[stage](ctx)
                markers.append(marker)
                status.done.add((stage, video_id))
                print(f"      {stage:<16} {marker['rows']:>4}/{marker['images']} khoá "
                      f"· {marker['boxes']:>5} hộp · {marker['elapsed_s']:>6.0f}s "
                      f"({marker['seconds_per_image']:.2f}s/ảnh)")

            # merge=True — BỐN notebook cùng ghi file manifest này.
            writer.write_part_manifest(video_id, markers,
                                       keyframes=len(keyframes), merge=True)
            done_count += 1
            images_seen += len(keyframes)

            elapsed = time.perf_counter() - started
            per_video = elapsed / done_count
            left_here = len(pending) - done_count
            left_all = len([v for v in MY_VIDEOS if stages_for(v)])
            print(f"  [{done_count:>3}] {video_id} xong | "
                  f"{per_video / 60:.1f} phút/video · "
                  f"còn ~{left_all * per_video / 3600:.1f}h cho cả phần của bạn")
        except Exception as err:
            failures.append((video_id, repr(err)))
            print(f"  [LỖI] {video_id}: {err}")
            traceback.print_exc(limit=3)
        finally:
            shutil.rmtree(WORK / "keyframes" / video_id, ignore_errors=True)
            (WORK / "map-keyframes" / f"{video_id}.csv").unlink(missing_ok=True)
            # KHÔNG `unload_models()` ở đây — xem ô mô tả đầu notebook. Thả
            # PaddleOCR rồi dựng lại trong cùng phiên là `PDX has already been
            # initialized`, và nó sẽ giết mọi video còn lại chứ không chỉ video này.

    if kf_zip is not None and kf_zip.parent == ZIPS:
        kf_zip.unlink(missing_ok=True)
        print(f"    đã xoá {zip_name}, đĩa trống {free_gb():.1f} GB")

print()
print("=" * 72)
elapsed = time.perf_counter() - started
print(f"phiên này: {done_count} video, {images_seen:,} ảnh, {len(failures)} lỗi, "
      f"{elapsed / 3600:.1f}h")
if images_seen:
    print(f"tốc độ   : {elapsed / images_seen:.2f}s/ảnh (cả hai tầng cộng lại)")
for video_id, err in failures:
    print(f"  {video_id}: {err}")

## Step 3 — chốt phiên

In [ ]:
unload_models()

fresh = StageStatus.load(transport, layout, MEMBER)
left  = [(s, v) for v in MY_VIDEOS for s in STAGE_LIST if not fresh.is_done(s, v)]

total = len(MY_VIDEOS) * len(STAGE_LIST)
print(f"{MEMBER}: {total - len(left)}/{total} việc đã lên {VOLUME}")
for stage in STAGE_LIST:
    n = sum(1 for v in MY_VIDEOS if fresh.is_done(stage, v))
    print(f"  {stage:<16} {n:>4}/{len(MY_VIDEOS)} video")

if left:
    print()
    print(f"còn {len(left)} việc — bấm Run All lại, nó chạy tiếp từ chỗ đứt")
else:
    print()
    print("XONG phần của bạn. Báo Lead.")
    print(f"  python scripts/verify_bundle.py --shards {SHARDS} --member {MEMBER}")

## Sau khi CẢ HAI người xong

```bash
python scripts/verify_bundle.py --shards config/ingest_shards_batch1_btc_ocr2.yaml
python scripts/merge_bundle.py  --shards config/ingest_shards_batch1_btc_ocr2.yaml
```

Lệnh gộp này chỉ đụng `paddle_ocr` + `viet_ocr` — nó lọc artifact theo `spec.stage in plan.stages`,
nên **không** ảnh hưởng lần gộp của lượt 2 (`ocr_text`, `sviptr_ocr`, `siglip_vectors`).

Kết quả: `data/vectors/paddle_ocr.json` và `data/vectors/viet_ocr.json`.

### Nạp vào Elasticsearch

Hai artifact **đã được cắm thành arm**: `paddle_ocr → source ocr_paddle → arm ocr_paddle`, và
`viet_ocr → ocr_vietocr → ocr_vietocr`. Nạp hai file phẳng vào text store:

```bash
python scripts/ingest_sota_text.py --vectors <bundle>/data/vectors     --sources ocr_paddle,ocr_vietocr
```

Sau đó hai arm tự xuất hiện trong `Cascade` — `build_text_arms` duyệt `TEXT_ARM_SPECS`, không
cần khai thêm ở đâu. Kiểm bằng `scripts/measure_arms.py`.

**Trọng số vẫn là 1.0 mặc định, và đó KHÔNG phải giá trị đã tune.** Bảng 40 keyframe ở đầu
notebook này biện minh cho việc *sinh* cả hai artifact; nó không đặt được trọng số. Trọng số đặt
bằng leo dốc trên tập **dev**, công bố trên **heldout** (§9.4) — và chỉ làm được sau khi lượt này
chạy xong, vì trước đó hai arm không có dữ liệu để mà đo.